# UrbanWatch — Dataset Structure Analysis

Run this notebook from the `UrbanWatch` project root. It only **reads and analyzes** the downloaded dataset folders; it does not move, rename, edit, or delete anything.

It is designed to inspect SpaceNet 2, LEVIR-CD+, and LoveDA so we can build the preprocessing pipeline from the actual files rather than assumptions.

In [2]:
from pathlib import Path
from collections import Counter

ROOT = Path.cwd()
print('Working directory:', ROOT)
print('\nTop-level contents:')
for p in sorted(ROOT.iterdir(), key=lambda x: (not x.is_dir(), x.name.lower())):
    print(('DIR ' if p.is_dir() else 'FILE'), p.name)


Working directory: c:\Users\shreyas\Documents\UrbanWatch

Top-level contents:
DIR  AOI_3_Paris_Test_public
DIR  AOI_3_Paris_Train
DIR  LEVIR-CD+
DIR  models
DIR  results
DIR  src
DIR  Test
DIR  Train
DIR  Val
FILE 01_dataset_structure_analysis.ipynb
FILE test_environment.py


## 1. Dataset folders found at the project root

In [3]:
keywords = ['spacenet', 'aoi_', 'paris', 'levir', 'loveda', 'train', 'val', 'test', 'building', 'change']
datasets = [p for p in ROOT.iterdir() if any(k in p.name.lower() for k in keywords)]

for p in datasets:
    print(f'{p.name} -> {p.resolve()}')

print(f'\nPotential dataset folders/files found: {len(datasets)}')

AOI_3_Paris_Test_public -> C:\Users\shreyas\Documents\UrbanWatch\AOI_3_Paris_Test_public
AOI_3_Paris_Train -> C:\Users\shreyas\Documents\UrbanWatch\AOI_3_Paris_Train
LEVIR-CD+ -> C:\Users\shreyas\Documents\UrbanWatch\LEVIR-CD+
Test -> C:\Users\shreyas\Documents\UrbanWatch\Test
test_environment.py -> C:\Users\shreyas\Documents\UrbanWatch\test_environment.py
Train -> C:\Users\shreyas\Documents\UrbanWatch\Train
Val -> C:\Users\shreyas\Documents\UrbanWatch\Val

Potential dataset folders/files found: 7


## 2. Folder tree (limited depth)

In [4]:
MAX_DEPTH = 3
MAX_FILES_PER_FOLDER = 20

def print_tree(path, prefix='', depth=0):
    if depth > MAX_DEPTH:
        return
    try:
        entries = sorted(path.iterdir(), key=lambda x: (not x.is_dir(), x.name.lower()))
    except Exception as e:
        print(prefix + f'[Could not read: {e}]')
        return

    dirs = [e for e in entries if e.is_dir()]
    files = [e for e in entries if e.is_file()]

    for d in dirs:
        print(prefix + '📁 ' + d.name)
        print_tree(d, prefix + '   ', depth + 1)

    for f in files[:MAX_FILES_PER_FOLDER]:
        print(prefix + '📄 ' + f.name)

    if len(files) > MAX_FILES_PER_FOLDER:
        print(prefix + f'   ... {len(files) - MAX_FILES_PER_FOLDER} more files')

for p in datasets:
    if p.is_dir():
        print('\n' + '=' * 80)
        print('TREE:', p.name)
        print('=' * 80)
        print_tree(p)


TREE: AOI_3_Paris_Test_public
📁 MUL
   📄 MUL_AOI_3_Paris_img1.tif
   📄 MUL_AOI_3_Paris_img1002.tif
   📄 MUL_AOI_3_Paris_img1014.tif
   📄 MUL_AOI_3_Paris_img1023.tif
   📄 MUL_AOI_3_Paris_img1024.tif
   📄 MUL_AOI_3_Paris_img1031.tif
   📄 MUL_AOI_3_Paris_img1033.tif
   📄 MUL_AOI_3_Paris_img1037.tif
   📄 MUL_AOI_3_Paris_img104.tif
   📄 MUL_AOI_3_Paris_img1044.tif
   📄 MUL_AOI_3_Paris_img1055.tif
   📄 MUL_AOI_3_Paris_img1056.tif
   📄 MUL_AOI_3_Paris_img1073.tif
   📄 MUL_AOI_3_Paris_img1084.tif
   📄 MUL_AOI_3_Paris_img1085.tif
   📄 MUL_AOI_3_Paris_img1087.tif
   📄 MUL_AOI_3_Paris_img1094.tif
   📄 MUL_AOI_3_Paris_img1096.tif
   📄 MUL_AOI_3_Paris_img11.tif
   📄 MUL_AOI_3_Paris_img1101.tif
      ... 361 more files
📁 MUL-PanSharpen
   📄 MUL-PanSharpen_AOI_3_Paris_img1.tif
   📄 MUL-PanSharpen_AOI_3_Paris_img1002.tif
   📄 MUL-PanSharpen_AOI_3_Paris_img1014.tif
   📄 MUL-PanSharpen_AOI_3_Paris_img1023.tif
   📄 MUL-PanSharpen_AOI_3_Paris_img1024.tif
   📄 MUL-PanSharpen_AOI_3_Paris_img1031.tif
   📄 M

## 3. File counts, extensions, and total size

In [5]:
def analyze_files(path):
    counts = Counter()
    total_files = 0
    total_bytes = 0
    examples = {}

    for f in path.rglob('*'):
        if f.is_file():
            total_files += 1
            try:
                total_bytes += f.stat().st_size
            except OSError:
                pass
            ext = f.suffix.lower() if f.suffix else '[no extension]'
            counts[ext] += 1
            examples.setdefault(ext, str(f.relative_to(path)))

    print(f'Root: {path}')
    print(f'Files: {total_files:,}')
    print(f'Total size: {total_bytes / 1024**3:.2f} GB')
    print('\nExtension counts:')
    for ext, count in counts.most_common():
        print(f'  {ext:18} {count:>10,}   example: {examples[ext]}')

for p in datasets:
    if p.is_dir():
        print('\n' + '=' * 80)
        print('FILE ANALYSIS:', p.name)
        print('=' * 80)
        analyze_files(p)


FILE ANALYSIS: AOI_3_Paris_Test_public
Root: c:\Users\shreyas\Documents\UrbanWatch\AOI_3_Paris_Test_public
Files: 1,524
Total size: 3.75 GB

Extension counts:
  .tif                    1,524   example: MUL\MUL_AOI_3_Paris_img1.tif

FILE ANALYSIS: AOI_3_Paris_Train
Root: c:\Users\shreyas\Documents\UrbanWatch\AOI_3_Paris_Train
Files: 5,741
Total size: 11.32 GB

Extension counts:
  .tif                    4,592   example: MUL\MUL_AOI_3_Paris_img10.tif
  .geojson                1,148   example: geojson\buildings\buildings_AOI_3_Paris_img10.geojson
  .csv                        1   example: summaryData\AOI_3_Paris_Train_Building_Solutions.csv

FILE ANALYSIS: LEVIR-CD+
Root: c:\Users\shreyas\Documents\UrbanWatch\LEVIR-CD+
Files: 2,959
Total size: 3.53 GB

Extension counts:
  .png                    2,955   example: test\A\train_638.png
  [no extension]              4   example: .DS_Store

FILE ANALYSIS: Test
Root: c:\Users\shreyas\Documents\UrbanWatch\Test
Files: 1,796
Total size: 2.91 GB



## 4. Sample image dimensions

In [6]:
try:
    from PIL import Image
except Exception as e:
    Image = None
    print('Pillow unavailable:', e)

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

if Image is not None:
    for p in datasets:
        if not p.is_dir():
            continue
        images = [f for f in p.rglob('*') if f.is_file() and f.suffix.lower() in IMAGE_EXTS]
        print('\n' + '=' * 80)
        print('IMAGE SAMPLES:', p.name)
        print('=' * 80)
        print('Image files found:', f'{len(images):,}')
        for f in images[:12]:
            try:
                with Image.open(f) as im:
                    print(f'{f.relative_to(p)} | size={im.size} | mode={im.mode} | format={im.format}')
            except Exception as e:
                print(f'{f.relative_to(p)} | ERROR: {e}')

More samples per pixel than can be decoded: 8
More samples per pixel than can be decoded: 8
More samples per pixel than can be decoded: 8
More samples per pixel than can be decoded: 8
More samples per pixel than can be decoded: 8
More samples per pixel than can be decoded: 8
More samples per pixel than can be decoded: 8
More samples per pixel than can be decoded: 8
More samples per pixel than can be decoded: 8
More samples per pixel than can be decoded: 8
More samples per pixel than can be decoded: 8
More samples per pixel than can be decoded: 8



IMAGE SAMPLES: AOI_3_Paris_Test_public
Image files found: 1,524
MUL\MUL_AOI_3_Paris_img1.tif | ERROR: cannot identify image file 'c:\\Users\\shreyas\\Documents\\UrbanWatch\\AOI_3_Paris_Test_public\\MUL\\MUL_AOI_3_Paris_img1.tif'
MUL\MUL_AOI_3_Paris_img1002.tif | ERROR: cannot identify image file 'c:\\Users\\shreyas\\Documents\\UrbanWatch\\AOI_3_Paris_Test_public\\MUL\\MUL_AOI_3_Paris_img1002.tif'
MUL\MUL_AOI_3_Paris_img1014.tif | ERROR: cannot identify image file 'c:\\Users\\shreyas\\Documents\\UrbanWatch\\AOI_3_Paris_Test_public\\MUL\\MUL_AOI_3_Paris_img1014.tif'
MUL\MUL_AOI_3_Paris_img1023.tif | ERROR: cannot identify image file 'c:\\Users\\shreyas\\Documents\\UrbanWatch\\AOI_3_Paris_Test_public\\MUL\\MUL_AOI_3_Paris_img1023.tif'
MUL\MUL_AOI_3_Paris_img1024.tif | ERROR: cannot identify image file 'c:\\Users\\shreyas\\Documents\\UrbanWatch\\AOI_3_Paris_Test_public\\MUL\\MUL_AOI_3_Paris_img1024.tif'
MUL\MUL_AOI_3_Paris_img1031.tif | ERROR: cannot identify image file 'c:\\Users\\shreya

More samples per pixel than can be decoded: 8
More samples per pixel than can be decoded: 8
More samples per pixel than can be decoded: 8
More samples per pixel than can be decoded: 8
More samples per pixel than can be decoded: 8
More samples per pixel than can be decoded: 8
More samples per pixel than can be decoded: 8
More samples per pixel than can be decoded: 8
More samples per pixel than can be decoded: 8
More samples per pixel than can be decoded: 8
More samples per pixel than can be decoded: 8
More samples per pixel than can be decoded: 8



IMAGE SAMPLES: AOI_3_Paris_Train
Image files found: 4,592
MUL\MUL_AOI_3_Paris_img10.tif | ERROR: cannot identify image file 'c:\\Users\\shreyas\\Documents\\UrbanWatch\\AOI_3_Paris_Train\\MUL\\MUL_AOI_3_Paris_img10.tif'
MUL\MUL_AOI_3_Paris_img100.tif | ERROR: cannot identify image file 'c:\\Users\\shreyas\\Documents\\UrbanWatch\\AOI_3_Paris_Train\\MUL\\MUL_AOI_3_Paris_img100.tif'
MUL\MUL_AOI_3_Paris_img1000.tif | ERROR: cannot identify image file 'c:\\Users\\shreyas\\Documents\\UrbanWatch\\AOI_3_Paris_Train\\MUL\\MUL_AOI_3_Paris_img1000.tif'
MUL\MUL_AOI_3_Paris_img1001.tif | ERROR: cannot identify image file 'c:\\Users\\shreyas\\Documents\\UrbanWatch\\AOI_3_Paris_Train\\MUL\\MUL_AOI_3_Paris_img1001.tif'
MUL\MUL_AOI_3_Paris_img1003.tif | ERROR: cannot identify image file 'c:\\Users\\shreyas\\Documents\\UrbanWatch\\AOI_3_Paris_Train\\MUL\\MUL_AOI_3_Paris_img1003.tif'
MUL\MUL_AOI_3_Paris_img1007.tif | ERROR: cannot identify image file 'c:\\Users\\shreyas\\Documents\\UrbanWatch\\AOI_3_Pari

## 5. Annotation clues

This checks small JSON/GeoJSON/CSV/XML/TXT/YAML files and prints a short sample. Large binary files are not opened.

In [7]:
TEXT_EXTS = {'.json', '.geojson', '.csv', '.txt', '.xml', '.yaml', '.yml'}
MAX_READ_BYTES = 1_000_000

for p in datasets:
    if not p.is_dir():
        continue
    candidates = [f for f in p.rglob('*') if f.is_file() and f.suffix.lower() in TEXT_EXTS]
    if not candidates:
        continue

    print('\n' + '=' * 80)
    print('ANNOTATION / METADATA SAMPLES:', p.name)
    print('=' * 80)
    for f in candidates[:15]:
        print('\n---', f.relative_to(p), '---')
        try:
            text = f.read_bytes()[:MAX_READ_BYTES].decode('utf-8', errors='replace')
            print(text[:2000].replace('\x00', ' '))
        except Exception as e:
            print('ERROR:', e)


ANNOTATION / METADATA SAMPLES: AOI_3_Paris_Train

--- geojson\buildings\buildings_AOI_3_Paris_img10.geojson ---
{
"type": "FeatureCollection",
"crs": { "type": "name", "properties": { "name": "urn:ogc:def:crs:OGC:1.3:CRS84" } },
                                                                                
"features": [

]
}


--- geojson\buildings\buildings_AOI_3_Paris_img100.geojson ---
{
"type": "FeatureCollection",
"crs": { "type": "name", "properties": { "name": "urn:ogc:def:crs:OGC:1.3:CRS84" } },
                                                                                
"features": [
{ "type": "Feature", "properties": { "OBJECTID_1": 0, "Name_1": "None", "AREA_1": 0.000000, "Shape_Leng": 0.000000, "Shape_Le_1": 0.001890, "Shape_Area": 0.000000, "partialBuilding": 1.000000, "partialDec": 0.078880 }, "geometry": { "type": "Polygon", "coordinates": [ [ [ 2.210981399959864, 49.023343219010123, 0.0 ], [ 2.210886256000038, 49.023391452000055, 0.0 ], [ 2.210981399959864, 49.02

In [8]:
from pathlib import Path
from collections import Counter

ROOT = Path.cwd()

dataset_roots = [
    ROOT / "AOI_3_Paris_Train",
    ROOT / "AOI_3_Paris_Test_public",
    ROOT / "LEVIR-CD+",
    ROOT / "Train",
    ROOT / "Val",
    ROOT / "Test",
]

for root in dataset_roots:

    if not root.exists():
        print(f"\nNOT FOUND: {root}")
        continue

    print("\n" + "=" * 70)
    print(root.name)
    print("=" * 70)

    # Immediate folders
    folders = [x for x in root.iterdir() if x.is_dir()]
    print("\nImmediate folders:")
    for x in folders:
        print("  ", x.name)

    # Recursive extension counts
    counts = Counter(
        x.suffix.lower()
        for x in root.rglob("*")
        if x.is_file()
    )

    print("\nFile types:")
    for ext, count in counts.most_common():
        print(f"  {ext or '[no extension]':15} {count:,}")

    # Important directory counts
    print("\nSubfolder file counts:")

    for folder in sorted(folders):
        count = sum(1 for x in folder.rglob("*") if x.is_file())
        print(f"  {folder.name:30} {count:,}")


AOI_3_Paris_Train

Immediate folders:
   geojson
   MUL
   MUL-PanSharpen
   PAN
   RGB-PanSharpen
   summaryData

File types:
  .tif            4,592
  .geojson        1,148
  .csv            1

Subfolder file counts:
  geojson                        1,148
  MUL                            1,148
  MUL-PanSharpen                 1,148
  PAN                            1,148
  RGB-PanSharpen                 1,148
  summaryData                    1

AOI_3_Paris_Test_public

Immediate folders:
   MUL
   MUL-PanSharpen
   PAN
   RGB-PanSharpen

File types:
  .tif            1,524

Subfolder file counts:
  MUL                            381
  MUL-PanSharpen                 381
  PAN                            381
  RGB-PanSharpen                 381

LEVIR-CD+

Immediate folders:
   test
   train

File types:
  .png            2,955
  [no extension]  4

Subfolder file counts:
  test                           1,045
  train                          1,913

Train

Immediate folders:
   Rural
   

In [9]:
from pathlib import Path

ROOT = Path.cwd()

for name in ["LEVIR-CD+", "Train", "Val", "Test"]:
    root = ROOT / name

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    if not root.exists():
        print("NOT FOUND")
        continue

    print("\nImmediate folders:")
    for p in sorted(root.iterdir()):
        print(" ", "DIR " if p.is_dir() else "FILE", p.name)

    print("\nSample files from each immediate folder:")

    for p in sorted(root.iterdir()):
        if p.is_dir():
            files = [x for x in p.rglob("*") if x.is_file()]

            print(f"\n[{p.name}]")
            print("File count:", len(files))

            for f in files[:5]:
                print(" ", f.relative_to(root))


LEVIR-CD+

Immediate folders:
  FILE .DS_Store
  DIR  test
  DIR  train

Sample files from each immediate folder:

[test]
File count: 1045
  test\.DS_Store
  test\A\train_638.png
  test\A\train_639.png
  test\A\train_640.png
  test\A\train_641.png

[train]
File count: 1913
  train\.DS_Store
  train\A\.DS_Store
  train\A\train_1.png
  train\A\train_10.png
  train\A\train_100.png

Train

Immediate folders:
  DIR  Rural
  DIR  Urban

Sample files from each immediate folder:

[Rural]
File count: 2732
  Rural\images_png\0.png
  Rural\images_png\1.png
  Rural\images_png\10.png
  Rural\images_png\100.png
  Rural\images_png\1000.png

[Urban]
File count: 2312
  Urban\images_png\1366.png
  Urban\images_png\1367.png
  Urban\images_png\1368.png
  Urban\images_png\1369.png
  Urban\images_png\1370.png

Val

Immediate folders:
  DIR  Rural
  DIR  Urban

Sample files from each immediate folder:

[Rural]
File count: 1984
  Rural\images_png\2522.png
  Rural\images_png\2523.png
  Rural\images_png\2524.p

In [10]:
from pathlib import Path

ROOT = Path.cwd()

targets = {
    "LEVIR-CD+": ROOT / "LEVIR-CD+",
    "LoveDA_Train": ROOT / "Train",
    "LoveDA_Val": ROOT / "Val",
    "LoveDA_Test": ROOT / "Test",
}

for name, root in targets.items():

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    if not root.exists():
        print("NOT FOUND")
        continue

    # Show directories up to 2 levels deep
    for current in sorted(
        [p for p in root.rglob("*") if p.is_dir()],
        key=lambda x: str(x).lower()
    ):

        relative = current.relative_to(root)

        # only show root + first two directory levels
        if len(relative.parts) <= 2:
            files = [f for f in current.iterdir() if f.is_file()]

            print(f"\n📁 {relative}")
            print(f"   Files directly inside: {len(files)}")

            for f in sorted(files)[:8]:
                print(f"   └── {f.name}")

    print("\nSample files:")
    all_files = [f for f in root.rglob("*") if f.is_file()]

    for f in all_files[:20]:
        print(" ", f.relative_to(root))


LEVIR-CD+

📁 test
   Files directly inside: 1
   └── .DS_Store

📁 test\A
   Files directly inside: 348
   └── train_638.png
   └── train_639.png
   └── train_640.png
   └── train_641.png
   └── train_642.png
   └── train_643.png
   └── train_644.png
   └── train_645.png

📁 test\B
   Files directly inside: 348
   └── train_638.png
   └── train_639.png
   └── train_640.png
   └── train_641.png
   └── train_642.png
   └── train_643.png
   └── train_644.png
   └── train_645.png

📁 test\label
   Files directly inside: 348
   └── train_638.png
   └── train_639.png
   └── train_640.png
   └── train_641.png
   └── train_642.png
   └── train_643.png
   └── train_644.png
   └── train_645.png

📁 train
   Files directly inside: 1
   └── .DS_Store

📁 train\A
   Files directly inside: 638
   └── .DS_Store
   └── train_1.png
   └── train_10.png
   └── train_100.png
   └── train_101.png
   └── train_102.png
   └── train_103.png
   └── train_104.png

📁 train\B
   Files directly inside: 637
   └── trai

In [11]:
from pathlib import Path
from PIL import Image
import numpy as np
import json

ROOT = Path(__file__).resolve().parent

print("=" * 70)
print("URBANWATCH DATASET SANITY CHECK")
print("=" * 70)

# ============================================================
# 1. LEVIR-CD+
# ============================================================

print("\n" + "=" * 70)
print("1. LEVIR-CD+")
print("=" * 70)

levir = ROOT / "LEVIR-CD+"

a = levir / "train" / "A" / "train_1.png"
b = levir / "train" / "B" / "train_1.png"
label = levir / "train" / "label" / "train_1.png"

for name, path in [
    ("Image A", a),
    ("Image B", b),
    ("Change Label", label),
]:
    print(f"\n{name}")
    print("Path:", path)
    print("Exists:", path.exists())

    if path.exists():
        with Image.open(path) as img:
            arr = np.array(img)

        print("Shape:", arr.shape)
        print("Data type:", arr.dtype)
        print("Min:", arr.min())
        print("Max:", arr.max())

        if name == "Change Label":
            print("Unique values:", np.unique(arr))


# ============================================================
# 2. LoveDA
# ============================================================

print("\n" + "=" * 70)
print("2. LoveDA")
print("=" * 70)

loveda_image = ROOT / "Train" / "Rural" / "images_png" / "0.png"
loveda_mask = ROOT / "Train" / "Rural" / "masks_png" / "0.png"

for name, path in [
    ("Image", loveda_image),
    ("Mask", loveda_mask),
]:
    print(f"\n{name}")
    print("Path:", path)
    print("Exists:", path.exists())

    if path.exists():
        with Image.open(path) as img:
            arr = np.array(img)

        print("Shape:", arr.shape)
        print("Data type:", arr.dtype)
        print("Min:", arr.min())
        print("Max:", arr.max())
        print("Unique values:", np.unique(arr)[:30])


# ============================================================
# 3. SpaceNet 2
# ============================================================

print("\n" + "=" * 70)
print("3. SpaceNet 2")
print("=" * 70)

spacenet = ROOT / "AOI_3_Paris_Train"

# RGB-PanSharpen image
rgb_folder = spacenet / "RGB-PanSharpen"

tif_files = sorted(rgb_folder.glob("*.tif"))

print("\nRGB-PanSharpen TIFF count:", len(tif_files))

if tif_files:
    image_path = tif_files[0]

    print("\nSample TIFF:")
    print("Path:", image_path)
    print("Size:", image_path.stat().st_size / (1024 ** 2), "MB")

# Matching GeoJSON
geojson_folder = spacenet / "geojson" / "buildings"

geojson_files = sorted(geojson_folder.glob("*.geojson"))

print("\nBuilding GeoJSON count:", len(geojson_files))

if geojson_files:
    geojson_path = geojson_files[0]

    print("\nSample GeoJSON:")
    print("Path:", geojson_path)

    with open(geojson_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("Type:", data.get("type"))
    print("CRS:", data.get("crs"))
    print("Number of features:", len(data.get("features", [])))

    if data.get("features"):
        feature = data["features"][0]

        print("\nFirst feature:")
        print("Geometry type:", feature["geometry"]["type"])
        print("Properties:", feature["properties"])


# ============================================================
# 4. Summary
# ============================================================

print("\n" + "=" * 70)
print("SANITY CHECK COMPLETE")
print("=" * 70)from pathlib import Path
from PIL import Image
import numpy as np
import json

ROOT = Path(__file__).resolve().parent

print("=" * 70)
print("URBANWATCH DATASET SANITY CHECK")
print("=" * 70)

# ============================================================
# 1. LEVIR-CD+
# ============================================================

print("\n" + "=" * 70)
print("1. LEVIR-CD+")
print("=" * 70)

levir = ROOT / "LEVIR-CD+"

a = levir / "train" / "A" / "train_1.png"
b = levir / "train" / "B" / "train_1.png"
label = levir / "train" / "label" / "train_1.png"

for name, path in [
    ("Image A", a),
    ("Image B", b),
    ("Change Label", label),
]:
    print(f"\n{name}")
    print("Path:", path)
    print("Exists:", path.exists())

    if path.exists():
        with Image.open(path) as img:
            arr = np.array(img)

        print("Shape:", arr.shape)
        print("Data type:", arr.dtype)
        print("Min:", arr.min())
        print("Max:", arr.max())

        if name == "Change Label":
            print("Unique values:", np.unique(arr))


# ============================================================
# 2. LoveDA
# ============================================================

print("\n" + "=" * 70)
print("2. LoveDA")
print("=" * 70)

loveda_image = ROOT / "Train" / "Rural" / "images_png" / "0.png"
loveda_mask = ROOT / "Train" / "Rural" / "masks_png" / "0.png"

for name, path in [
    ("Image", loveda_image),
    ("Mask", loveda_mask),
]:
    print(f"\n{name}")
    print("Path:", path)
    print("Exists:", path.exists())

    if path.exists():
        with Image.open(path) as img:
            arr = np.array(img)

        print("Shape:", arr.shape)
        print("Data type:", arr.dtype)
        print("Min:", arr.min())
        print("Max:", arr.max())
        print("Unique values:", np.unique(arr)[:30])


# ============================================================
# 3. SpaceNet 2
# ============================================================

print("\n" + "=" * 70)
print("3. SpaceNet 2")
print("=" * 70)

spacenet = ROOT / "AOI_3_Paris_Train"

# RGB-PanSharpen image
rgb_folder = spacenet / "RGB-PanSharpen"

tif_files = sorted(rgb_folder.glob("*.tif"))

print("\nRGB-PanSharpen TIFF count:", len(tif_files))

if tif_files:
    image_path = tif_files[0]

    print("\nSample TIFF:")
    print("Path:", image_path)
    print("Size:", image_path.stat().st_size / (1024 ** 2), "MB")

# Matching GeoJSON
geojson_folder = spacenet / "geojson" / "buildings"

geojson_files = sorted(geojson_folder.glob("*.geojson"))

print("\nBuilding GeoJSON count:", len(geojson_files))

if geojson_files:
    geojson_path = geojson_files[0]

    print("\nSample GeoJSON:")
    print("Path:", geojson_path)

    with open(geojson_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("Type:", data.get("type"))
    print("CRS:", data.get("crs"))
    print("Number of features:", len(data.get("features", [])))

    if data.get("features"):
        feature = data["features"][0]

        print("\nFirst feature:")
        print("Geometry type:", feature["geometry"]["type"])
        print("Properties:", feature["properties"])


# ============================================================
# 4. Summary
# ============================================================

print("\n" + "=" * 70)
print("SANITY CHECK COMPLETE")
print("=" * 70)from pathlib import Path
from PIL import Image
import numpy as np
import json

ROOT = Path(__file__).resolve().parent

print("=" * 70)
print("URBANWATCH DATASET SANITY CHECK")
print("=" * 70)

# ============================================================
# 1. LEVIR-CD+
# ============================================================

print("\n" + "=" * 70)
print("1. LEVIR-CD+")
print("=" * 70)

levir = ROOT / "LEVIR-CD+"

a = levir / "train" / "A" / "train_1.png"
b = levir / "train" / "B" / "train_1.png"
label = levir / "train" / "label" / "train_1.png"

for name, path in [
    ("Image A", a),
    ("Image B", b),
    ("Change Label", label),
]:
    print(f"\n{name}")
    print("Path:", path)
    print("Exists:", path.exists())

    if path.exists():
        with Image.open(path) as img:
            arr = np.array(img)

        print("Shape:", arr.shape)
        print("Data type:", arr.dtype)
        print("Min:", arr.min())
        print("Max:", arr.max())

        if name == "Change Label":
            print("Unique values:", np.unique(arr))


# ============================================================
# 2. LoveDA
# ============================================================

print("\n" + "=" * 70)
print("2. LoveDA")
print("=" * 70)

loveda_image = ROOT / "Train" / "Rural" / "images_png" / "0.png"
loveda_mask = ROOT / "Train" / "Rural" / "masks_png" / "0.png"

for name, path in [
    ("Image", loveda_image),
    ("Mask", loveda_mask),
]:
    print(f"\n{name}")
    print("Path:", path)
    print("Exists:", path.exists())

    if path.exists():
        with Image.open(path) as img:
            arr = np.array(img)

        print("Shape:", arr.shape)
        print("Data type:", arr.dtype)
        print("Min:", arr.min())
        print("Max:", arr.max())
        print("Unique values:", np.unique(arr)[:30])


# ============================================================
# 3. SpaceNet 2
# ============================================================

print("\n" + "=" * 70)
print("3. SpaceNet 2")
print("=" * 70)

spacenet = ROOT / "AOI_3_Paris_Train"

# RGB-PanSharpen image
rgb_folder = spacenet / "RGB-PanSharpen"

tif_files = sorted(rgb_folder.glob("*.tif"))

print("\nRGB-PanSharpen TIFF count:", len(tif_files))

if tif_files:
    image_path = tif_files[0]

    print("\nSample TIFF:")
    print("Path:", image_path)
    print("Size:", image_path.stat().st_size / (1024 ** 2), "MB")

# Matching GeoJSON
geojson_folder = spacenet / "geojson" / "buildings"

geojson_files = sorted(geojson_folder.glob("*.geojson"))

print("\nBuilding GeoJSON count:", len(geojson_files))

if geojson_files:
    geojson_path = geojson_files[0]

    print("\nSample GeoJSON:")
    print("Path:", geojson_path)

    with open(geojson_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("Type:", data.get("type"))
    print("CRS:", data.get("crs"))
    print("Number of features:", len(data.get("features", [])))

    if data.get("features"):
        feature = data["features"][0]

        print("\nFirst feature:")
        print("Geometry type:", feature["geometry"]["type"])
        print("Properties:", feature["properties"])


# ============================================================
# 4. Summary
# ============================================================

print("\n" + "=" * 70)
print("SANITY CHECK COMPLETE")
print("=" * 70)

SyntaxError: invalid syntax (1939216093.py, line 136)

## 6. What we need from this analysis

**SpaceNet 2:** identify the imagery files, building-label files, and train/test organization.

**LEVIR-CD+:** identify the two temporal image folders/pairs and the change-mask organization.

**LoveDA:** identify the image and label folders for Train/Val/Test and the label encoding.

After running all cells, save the notebook. The outputs will tell us exactly how to build the loaders and preprocessing scripts.